# Orbit Wars: Raw Replay Downloader
This notebook searches through all Kaggle datasets for replays won by a specific player, and packages the raw `.json` files into a single `.zip` file so you can download them and process them locally.

In [ ]:
import os
import json
import shutil
import zipfile
from pathlib import Path

# ---------------------------------------------------------
# CONFIGURATION: Set the player you want to download here!
TARGET_PLAYER = "Isaiah @ Tufa Labs"
# ---------------------------------------------------------

INPUT_DIR = Path("/kaggle/input")
OUTPUT_ZIP = Path("/kaggle/working/Target_Player_Replays.zip")

print(f"Scanning for all .json replays...")
replays = list(INPUT_DIR.rglob("*.json"))
print(f"Found {len(replays)} replays across all datasets.")

In [ ]:
print(f"Beginning search. Filtering exclusively for wins by: {TARGET_PLAYER}")

collected_count = 0

# Open a zip file to stream the json files directly into it (saving disk space)
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for i, path in enumerate(replays):
        
        # Optimization 1: Fast text check before parsing JSON
        try:
            with open(path, 'r', encoding='utf-8') as f:
                head = f.read(10000)
                if TARGET_PLAYER not in head:
                    continue
        except:
            continue
            
        # Optimization 2: Ensure the player actually WON the match
        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            team_names = data.get("info", {}).get("TeamNames", [])
            if TARGET_PLAYER not in team_names:
                continue
                
            rewards = data.get('rewards', []) or []
            rewards_num = [r for r in rewards if r is not None]
            if not rewards_num:
                continue
                
            max_r = max(rewards_num)
            if rewards_num.count(max_r) > 1 or max_r <= 0:
                continue # Tie or invalid
                
            winner_idx = rewards.index(max_r)
            if winner_idx >= len(team_names) or team_names[winner_idx] != TARGET_PLAYER:
                continue # They played but didn't win
                
            # If we reach here, they won! Add the file to the zip.
            zipf.write(path, arcname=f"{TARGET_PLAYER.replace(' ', '_')}_win_{collected_count}.json")
            collected_count += 1
            
        except Exception as e:
            continue
            
        if i % 5000 == 0 and i > 0:
            print(f"Scanned {i} / {len(replays)} replays... Found {collected_count} wins so far.")

print("\n=========================================")
print("SEARCH COMPLETE!")
print(f"Successfully packaged {collected_count} winning replays.")
print(f"You can now download: {OUTPUT_ZIP}")
print("=========================================")